In [2]:
import pandas as pd
df = pd.read_csv(r'F:\Rehab work\NFL Hackathon\Raw Data\nfl-big-data-bowl-2027\player_play.csv')
df


,game_id,play_id,team_abbr,nfl_id,lined_up_position,play_description,quarter,down,yards_to_go,possession_team,...,sack,tackle,assist,caused_forced_fumble,recovered_fumble,tackle_for_loss,time_to_pressure,time_to_qb_hurry,coverage_assignment,coverage_assignment_at_snap
0,2025111610,2279,BAL,57183,T,(8:42) (Shotgun) L.Jackson pass incomplete dee...,3,1,10,BAL,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026011701,1626,SF,58248,INTERIOR_LINE,(3:48) (Shotgun) K.Walker left end to SEA 28 f...,2,2,7,SEA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2025101900,2838,LAR,57140,EDGE,(7:01) C.Van Lanen reported in as eligible. T...,3,2,2,JAX,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NO_ASSIGNMENT,NO_ASSIGNMENT
3,2025111300,2628,NYJ,58209,T,(4:05) I.Williams left end to NE 22 for 4 yard...,3,1,10,NYJ,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2025121406,2653,NYG,57191,OLB,"(10:08) (No Huddle, Shotgun) M.Mariota scrambl...",3,2,10,WAS,...,0.0,1.0,0.0,0.0,NaN,False,NaN,NaN,HOOK_CURL_LEFT,CURL_FLAT_LEFT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
314192,2025111608,1604,SF,57152,TE,(6:20) B.Purdy pass incomplete short middle to...,2,1,10,SF,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
314193,2026011200,150,PIT,57172,C,(12:08) (Shotgun) A.Rodgers pass incomplete sh...,1,3,7,PIT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
314194,2025081659,537,NYJ,58209,T,(6:20) B.Hall right guard to 50 for 4 yards (R...,1,2,5,NYJ,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
314195,2025082359,1861,LAC,56096,INTERIOR_LINE,(4:07) (Shotgun) I.Guerendo right end to 50 fo...,2,2,2,SF,...,NaN,1.0,0.0,0.0,NaN,False,NaN,NaN,NaN,NaN


In [3]:
print("Columns 1–25:")
print(df.columns[:25].tolist())

Columns 1–25:
['game_id', 'play_id', 'team_abbr', 'nfl_id', 'lined_up_position', 'play_description', 'quarter', 'down', 'yards_to_go', 'possession_team', 'yardline_side', 'yardline_number', 'game_clock', 'pre_snap_home_score', 'pre_snap_visitor_score', 'play_direction', 'pre_snap_home_team_win_probability', 'pre_snap_visitor_team_win_probability', 'home_team_win_probability_added', 'visitor_team_win_probility_added', 'expected_points_added', 'expected_points', 'offense_formation', 'receiver_alignment', 'pass_result']


In [4]:
print("Columns 26–50:")
print(df.columns[25:50].tolist())

Columns 26–50:
['pass_length', 'play_nullified_by_penalty', 'penalty_yards', 'pre_penalty_yards_gained', 'yards_gained', 'team_coverage_man_zone', 'team_coverage_type', 'in_motion_at_ball_snap', 'target', 'rec_yards', 'rush_yards', 'yards_after_catch', 'fumble', 'fumble_lost', 'route_ran', 'separation_at_pass_forward', 'cushion', 'expected_yards_after_catch', 'dropback_duration', 'extended_sack_allowed', 'pass_rushers_encountered', 'peak_pressure_probability_allowed', 'pressure_allowed', 'sack_allowed', 'time_to_pressure_allowed']


In [5]:
print("Columns 51–64:")
print(df.columns[50:].tolist())

Columns 51–64:
['quick_pressure', 'unblocked_pressure', 'blitzing', 'player_get_off', 'sack', 'tackle', 'assist', 'caused_forced_fumble', 'recovered_fumble', 'tackle_for_loss', 'time_to_pressure', 'time_to_qb_hurry', 'coverage_assignment', 'coverage_assignment_at_snap']


### Step 1 — Dataset Structure and Key Validation

**Objective:** Understand the granularity of `player_play.csv` and validate the candidate composite key.

**Checks:**
- Total records and columns
- Duplicate records
- Composite key uniqueness (`game_id`, `play_id`, `nfl_id`)
- Missing values in key columns
- Number of unique games, plays, and players

**Expected Grain:** One record per player per play within a game.

**Purpose:** Confirm that player-play records can be uniquely identified before proceeding with data quality and WR-specific analysis.

In [6]:
key_cols = ["game_id", "play_id", "nfl_id"]

print("Dataset Shape:", df.shape)

print("\nDuplicate Rows:", df.duplicated().sum())

print(
    "Duplicate Composite Keys:",
    df.duplicated(subset=key_cols).sum()
)

print("\nMissing Values in Composite Key:")
print(df[key_cols].isna().sum())

print("\nUnique Games:", df["game_id"].nunique())

print(
    "Unique Game-Plays:",
    df[["game_id", "play_id"]].drop_duplicates().shape[0]
)

print("Unique Players:", df["nfl_id"].nunique())

Dataset Shape: (314197, 64)

Duplicate Rows: 0
Duplicate Composite Keys: 0

Missing Values in Composite Key:
game_id    0
play_id    0
nfl_id     0
dtype: int64

Unique Games: 1001
Unique Game-Plays: 114954
Unique Players: 497


In [7]:
players_df = pd.read_csv(r"F:\Rehab work\NFL Hackathon\Raw Data\nfl-big-data-bowl-2027\players.csv")

players_ids = set(players_df["nfl_id"])
player_play_ids = set(df["nfl_id"])

missing_from_player_play = players_ids - player_play_ids
missing_from_players = player_play_ids - players_ids

print(
    "Players without player_play records:",
    len(missing_from_player_play)
)

print(
    "player_play IDs not found in players:",
    len(missing_from_players)
)

# Display unmatched player details
display(
    players_df[
        players_df["nfl_id"].isin(missing_from_player_play)
    ][["nfl_id", "display_name", "nfl_position", "draft_year"]]
)

Players without player_play records: 13
player_play IDs not found in players: 0


,nfl_id,display_name,nfl_position,draft_year
132,56648,Ikenna Enechukwu,OLB,2023
134,56363,Leonard Taylor,TE,2023
307,57743,Trajan Jeffcoat,DE,2024
335,57660,Lideatrick Griffin,WR,2024
376,58293,Caleb Ransaw,FS,2025
394,58326,David Walker,OLB,2025
424,58374,Zah Frazier,CB,2025
439,58407,Gavin Bartholomew,TE,2025
445,58417,Robert Longerbeam,CB,2025
459,58455,John Williams,G,2025


### Referential Integrity Findings — player_play.csv

- All 497 unique player IDs in `player_play.csv` exist in `players.csv`.
- 13 players from `players.csv` have no corresponding records in `player_play.csv`.
- One of these players is a Wide Receiver: Lideatrick Griffin (`nfl_id = 57660`).
- Therefore, 105 of the 106 WR players have records in `player_play.csv`.

**Assessment:** No orphan player IDs were identified in `player_play.csv`. The 13 unmatched players represent a coverage gap, not necessarily a data quality error.

**Decision:** Preserve all players in the reference dataset and document the difference in player coverage. Do not create artificial play records for unmatched players.

In [8]:
combine_df = pd.read_csv(r"F:\Rehab work\NFL Hackathon\Raw Data\nfl-big-data-bowl-2027\combine_results.csv")

# Players without player_play records
missing_ids = set(players_df["nfl_id"]) - set(df["nfl_id"])

# Get their information from players.csv
missing_players = players_df[
    players_df["nfl_id"].isin(missing_ids)
].copy()

# Add draft status
missing_players["draft_status"] = missing_players[
    "draft_round"
].apply(lambda x: "Undrafted" if pd.isna(x) else "Drafted")

# Match with combine_results.csv
validation = missing_players.merge(
    combine_df[["nfl_id", "combine_position"]],
    on="nfl_id",
    how="left",
    indicator=True
)

display(validation[[
    "nfl_id",
    "display_name",
    "draft_year",
    "draft_status",
    "nfl_position",
    "combine_position",
    "_merge"
]])

,nfl_id,display_name,draft_year,draft_status,nfl_position,combine_position,_merge
0,56648,Ikenna Enechukwu,2023,Undrafted,OLB,DL,both
1,56363,Leonard Taylor,2023,Undrafted,TE,TE,both
2,57743,Trajan Jeffcoat,2024,Undrafted,DE,DL,both
3,57660,Lideatrick Griffin,2024,Undrafted,WR,WR,both
4,58293,Caleb Ransaw,2025,Drafted,FS,DB,both
5,58326,David Walker,2025,Drafted,OLB,DL,both
6,58374,Zah Frazier,2025,Drafted,CB,DB,both
7,58407,Gavin Bartholomew,2025,Drafted,TE,TE,both
8,58417,Robert Longerbeam,2025,Drafted,CB,DB,both
9,58455,John Williams,2025,Drafted,G,OL,both


### Cross-Table Player Coverage Findings

- All 497 player IDs in `player_play.csv` exist in `players.csv`.
- 13 players from `players.csv` have no corresponding player-play records.
- All 13 unmatched players are present in `combine_results.csv`.
- Among these 13 players, 6 are Drafted and 7 are Undrafted.
- One unmatched player is a Wide Receiver: Lideatrick Griffin (`nfl_id = 57660`).
- The WR population decreases from 106 to 105 when restricting the analysis to players with player-play records.

**Conclusion:** The unmatched records represent differences in dataset coverage rather than confirmed data quality errors. All original player records should be preserved.

### Step 3 — Missing Values Analysis

**Objective:** Evaluate data completeness across all 64 columns in `player_play.csv`.

**Checks:**
- Missing value count per column
- Missing percentage per column
- Fully populated columns
- Columns with high missingness

**Important Consideration:** Missing values may be structurally expected because certain metrics apply only to specific player positions, play types, or game situations.

**Goal:** Distinguish expected missingness from potential data quality issues before preprocessing.

In [9]:
missing_summary = pd.DataFrame({
    "column_name": df.columns,
    "missing_count": df.isna().sum().values,
    "missing_percentage": (
        df.isna().mean().values * 100
    ).round(2)
})

missing_summary = missing_summary.sort_values(
    by="missing_percentage",
    ascending=False
)

print("Columns with missing values:",
      (missing_summary["missing_count"] > 0).sum())

print("Columns without missing values:",
      (missing_summary["missing_count"] == 0).sum())

display(missing_summary)

Columns with missing values: 45
Columns without missing values: 19


,column_name,missing_count,missing_percentage
35,rush_yards,314175,99.99
37,fumble,314124,99.98
38,fumble_lost,314124,99.98
58,recovered_fumble,311983,99.30
50,quick_pressure,310434,98.80
...,...,...,...
8,yards_to_go,0,0.00
11,yardline_number,0,0.00
12,game_clock,0,0.00
13,pre_snap_home_score,0,0.00


In [10]:
# Identify WR players
wr_ids = players_df.loc[
    players_df["nfl_position"] == "WR",
    "nfl_id"
]

# Filter player_play records
wr_df = df[
    df["nfl_id"].isin(wr_ids)
].copy()

print("WR Records:", len(wr_df))
print("Unique WR Players:", wr_df["nfl_id"].nunique())

# WR-specific completeness
wr_missing = pd.DataFrame({
    "missing_count": wr_df.isna().sum(),
    "missing_percentage": (
        wr_df.isna().mean() * 100
    ).round(2)
})

display(
    wr_missing.sort_values(
        "missing_percentage",
        ascending=False
    )
)

WR Records: 42806
Unique WR Players: 105


,missing_count,missing_percentage
unblocked_pressure,42806,100.0
quick_pressure,42806,100.0
sack,42806,100.0
player_get_off,42806,100.0
time_to_pressure,42806,100.0
...,...,...
yardline_number,0,0.0
game_clock,0,0.0
pre_snap_home_score,0,0.0
target,0,0.0


In [11]:
position_check = df.merge(
    players_df[["nfl_id", "nfl_position"]],
    on="nfl_id",
    how="left"
)

display(
    position_check.groupby("nfl_position")[
        "unblocked_pressure"
    ].agg(
        total_records="size",
        non_null_records="count"
    ).sort_values(
        "non_null_records",
        ascending=False
    )
)

,total_records,non_null_records
nfl_position,,
CB,44455,25731
SS,24962,14657
DE,23604,13004
OLB,21324,12217
DT,20790,11034
FS,14630,8604
NT,2517,1183
ILB,389,202
MLB,282,110


### Position-Dependent Missingness Analysis

**Objective:** Investigate whether missing values in `unblocked_pressure` are associated with player positions.

**Findings:**
- All 42,806 WR records have missing values in `unblocked_pressure` (100% missingness).
- The metric is primarily populated for defensive positions such as CB, SS, DE, OLB, DT, and FS.
- Offensive positions such as WR, TE, G, and T have no recorded values for this metric.
- A small number of non-null values (6 records) were observed for the Center (C) position and may require further investigation.

**Interpretation:**

The observed missingness pattern is strongly associated with player position.

Since `unblocked_pressure` is primarily a defensive pass-rushing metric, its missing values for WRs are likely **structural missingness (Not Applicable)** rather than data quality errors.

**Decision:**
- Preserve missing values in the original dataset.
- Do not apply imputation to this column.
- Consider excluding this feature from WR-specific modeling if it is not relevant to the analysis objective.

**Conclusion:** Missingness in `unblocked_pressure` is position-dependent. However, missingness in other columns must be evaluated individually before classifying it as structural.

In [12]:
fully_missing_wr = wr_missing[
    wr_missing["missing_percentage"] == 100
]

display(fully_missing_wr)

,missing_count,missing_percentage
rush_yards,42806,100.0
quick_pressure,42806,100.0
unblocked_pressure,42806,100.0
player_get_off,42806,100.0
sack,42806,100.0
time_to_pressure,42806,100.0
time_to_qb_hurry,42806,100.0
coverage_assignment,42806,100.0
coverage_assignment_at_snap,42806,100.0


### WR-Specific Fully Missing Columns

**Findings:**

Among 42,806 player-play records belonging to 105 Wide Receivers, 9 columns have 100% missing values.

**Likely position-dependent missingness:**

- `quick_pressure`
- `unblocked_pressure`
- `player_get_off`
- `sack`
- `time_to_pressure`
- `time_to_qb_hurry`
- `coverage_assignment`
- `coverage_assignment_at_snap`

These metrics are primarily associated with defensive responsibilities and are not generally applicable to the offensive WR role.

**Requires further investigation:**

- `rush_yards`: Although all WR records contain NULL values, Wide Receivers can participate in rushing plays. Therefore, this column should not automatically be classified as Not Applicable.

**Preprocessing Decision:**

- Preserve the original data.
- Avoid imputing structurally missing defensive metrics.
- Investigate `rush_yards` before making a preprocessing decision.
- Exclude irrelevant features only when preparing the final WR-specific analytical dataset.

In [13]:
rush_check = df.merge(
    players_df[["nfl_id", "nfl_position"]],
    on="nfl_id",
    how="left"
)

display(
    rush_check.groupby("nfl_position")["rush_yards"]
    .agg(
        total_records="size",
        non_null_records="count"
    )
    .sort_values("non_null_records", ascending=False)
)

,total_records,non_null_records
nfl_position,,
RB,86,22
C,15230,0
CB,44455,0
DE,23604,0
DB,186,0
FB,36,0
FS,14630,0
G,47882,0
DT,20790,0


### Investigation of `rush_yards` Missingness

**Findings:**

- All 42,806 WR records have NULL values in `rush_yards`.
- Only the RB position contains non-null values (22 out of 86 records).
- No other player position has recorded `rush_yards` values.

**Interpretation:**

The availability of `rush_yards` appears to be highly position-dependent within this dataset.

However, Wide Receivers can participate in rushing plays. Therefore, the absence of recorded values for WRs should not be interpreted as a football rule or definitive evidence that WR rushing never occurred.

**Decision:**

- Preserve the original NULL values.
- Do not replace missing `rush_yards` values with zero without confirming the dataset's recording rules.
- Exclude `rush_yards` from the initial WR-specific analysis if it provides no usable observations, while documenting this limitation.

In [14]:
partial_missing_wr = wr_missing[
    (wr_missing["missing_percentage"] > 0) &
    (wr_missing["missing_percentage"] < 100)
].sort_values(
    "missing_percentage",
    ascending=False
)

display(partial_missing_wr)

,missing_count,missing_percentage
time_to_pressure_allowed,42801,99.99
recovered_fumble,42785,99.95
fumble_lost,42762,99.90
fumble,42762,99.90
peak_pressure_probability_allowed,42763,99.90
pass_rushers_encountered,42718,99.79
sack_allowed,42718,99.79
pressure_allowed,42718,99.79
dropback_duration,42718,99.79
extended_sack_allowed,42718,99.79


In [15]:
receiving_check = (
    wr_df.groupby("target", dropna=False)
    .agg(
        total_records=("nfl_id", "size"),
        rec_yards_non_null=("rec_yards", "count"),
        yac_non_null=("yards_after_catch", "count"),
        expected_yac_non_null=("expected_yards_after_catch", "count")
    )
)

display(receiving_check)

,total_records,rec_yards_non_null,yac_non_null,expected_yac_non_null
target,,,,
False,34713,2,2,0
True,8093,5006,5006,7347


In [16]:
exceptions = wr_df[
    (wr_df["target"] == False) &
    (wr_df["rec_yards"].notna())
]

display(exceptions[[
    "game_id",
    "play_id",
    "nfl_id",
    "play_description",
    "target",
    "pass_result",
    "rec_yards",
    "yards_after_catch"
]])

,game_id,play_id,nfl_id,play_description,target,pass_result,rec_yards,yards_after_catch
250139,2024091511,4182,55928,(:09) (Shotgun) B.Nix pass deep right to J.Rey...,False,C,2.0,2.0
259609,2024110305,2105,57155,(:02) (Shotgun) J.Herbert pass short left to Q...,False,C,-6.0,-6.0


In [17]:
display(
    wr_df[wr_df["target"] == True]
    .groupby("pass_result", dropna=False)
    .agg(
        total_records=("nfl_id", "size"),
        rec_yards_non_null=("rec_yards", "count"),
        yac_non_null=("yards_after_catch", "count")
    )
)

,total_records,rec_yards_non_null,yac_non_null
pass_result,,,
C,5006,5006,5006
I,2855,0,0
IN,232,0,0


### Receiving Metrics — Missingness Validation

**Findings:**

Among 8,093 targeted WR records:

- 5,006 completed passes (`C`) have non-null `rec_yards` and `yards_after_catch`.
- 2,855 incomplete passes (`I`) have NULL receiving-yard values.
- 232 interceptions (`IN`) have NULL receiving-yard values.
- Two non-targeted WR records contain non-null receiving-yard values despite `target = False`.

**Interpretation:**

Missingness in `rec_yards` and `yards_after_catch` is strongly associated with pass completion status and is predominantly event-dependent rather than random.

The two non-targeted records with receiving-yard values represent potential inconsistencies requiring further investigation.

**Decision:**

- Preserve original NULL values.
- Do not automatically impute missing receiving-yard values with zero.
- Flag the two exceptions for later validation.
- Retain `pass_result` and `target` as important contextual features.

In [18]:
separation_cushion_check = (
    wr_df.groupby("target", dropna=False)
    .agg(
        total_records=("nfl_id", "size"),
        separation_missing=(
            "separation_at_pass_forward",
            lambda x: x.isna().sum()
        ),
        cushion_missing=(
            "cushion",
            lambda x: x.isna().sum()
        )
    )
)

separation_cushion_check["separation_missing_pct"] = (
    separation_cushion_check["separation_missing"]
    / separation_cushion_check["total_records"] * 100
).round(2)

separation_cushion_check["cushion_missing_pct"] = (
    separation_cushion_check["cushion_missing"]
    / separation_cushion_check["total_records"] * 100
).round(2)

display(separation_cushion_check)

,total_records,separation_missing,cushion_missing,separation_missing_pct,cushion_missing_pct
target,,,,,
False,34713,5281,1610,15.21,4.64
True,8093,0,465,0.00,5.75


### Separation and Cushion — Missingness Investigation

**Objective:** Investigate missing values in two important WR performance features:

- `separation_at_pass_forward`: Distance between the receiver and the nearest defender when the forward pass is released.
- `cushion`: Initial distance between the receiver and the defender before the snap.

**Initial Findings:**
- Separation: 5,281 missing records (12.34%).
- Cushion: 2,075 missing records (4.85%).

**Hypothesis:** Missingness may be associated with targeting status, passing situations, or route participation.

**Next Check:** Compare missing percentages between targeted and non-targeted WR records.

**Preprocessing Decision:** No imputation or row deletion until the missingness patterns are understood.

separation_by_pass = (
    wr_df.groupby("pass_result", dropna=False)
    .agg(
        total_records=("nfl_id", "size"),
        separation_missing=(
            "separation_at_pass_forward",
            lambda x: x.isna().sum()
        )
    )
)

separation_by_pass["missing_pct"] = (
    separation_by_pass["separation_missing"]
    / separation_by_pass["total_records"] * 100
).round(2)

display(separation_by_pass)

### Separation and Cushion — Missingness Findings

**Scope:** 42,806 WR player-play records.

**Separation Findings:**
- All 8,093 targeted records have non-null `separation_at_pass_forward`.
- Among 34,713 non-targeted records, 5,281 (15.21%) have missing separation values.
- The remaining 29,432 non-targeted records have valid separation measurements.

**Cushion Findings:**
- Non-targeted records: 1,610 missing values (4.64%).
- Targeted records: 465 missing values (5.75%).

**Interpretation:**

Separation missingness is associated with targeting status, but separation is also recorded for many non-targeted receivers. The exact reason for missing measurements remains unconfirmed.

Cushion missingness occurs in both targeted and non-targeted records, with relatively similar missing percentages.

**Preprocessing Decision:**
- Preserve missing values in both features.
- Do not replace missing distance measurements with zero.
- Investigate separation missingness using play context before selecting an imputation or exclusion strategy.

In [19]:
separation_by_pass = (
    wr_df.groupby("pass_result", dropna=False)
    .agg(
        total_records=("nfl_id", "size"),
        separation_missing=(
            "separation_at_pass_forward",
            lambda x: x.isna().sum()
        )
    )
)

separation_by_pass["missing_pct"] = (
    separation_by_pass["separation_missing"]
    / separation_by_pass["total_records"] * 100
).round(2)

display(separation_by_pass)

,total_records,separation_missing,missing_pct
pass_result,,,
C,23603,0,0.0
I,13006,0,0.0
IN,916,0,0.0
R,2321,2321,100.0
S,2960,2960,100.0


### Separation Missingness — Root Cause Analysis

**Objective:** Determine why `separation_at_pass_forward` contains 5,281 missing values among WR player-play records.

**Findings:**

| Pass Result | Total Records | Missing Separation | Missing % |
|---|---:|---:|---:|
| C — Complete Pass | 23,603 | 0 | 0% |
| I — Incomplete Pass | 13,006 | 0 | 0% |
| IN — Interception | 916 | 0 | 0% |
| R — Run | 2,321 | 2,321 | 100% |
| S — Sack | 2,960 | 2,960 | 100% |

**Interpretation:**

All 5,281 missing separation values occur in rushing or sack plays.

Since `separation_at_pass_forward` measures receiver-defender distance at the moment a forward pass is released, the metric is not applicable when no forward pass is thrown.

**Conclusion:**

Missingness in `separation_at_pass_forward` is fully explained by play outcome in the observed dataset and can be classified as **Structural / Event-Dependent Missingness**.

**Preprocessing Decision:**

- Preserve the original NULL values.
- Do not impute missing separation values with zero.
- Use records with valid separation measurements when analyzing separation at pass release.
- Document the applicable play types when calculating separation-based metrics.

In [20]:
cushion_by_pass = (
    wr_df.groupby("pass_result", dropna=False)
    .agg(
        total_records=("nfl_id", "size"),
        cushion_missing=("cushion", lambda x: x.isna().sum())
    )
)

cushion_by_pass["missing_pct"] = (
    cushion_by_pass["cushion_missing"]
    / cushion_by_pass["total_records"] * 100
).round(2)

display(cushion_by_pass)

,total_records,cushion_missing,missing_pct
pass_result,,,
C,23603,1270,5.38
I,13006,540,4.15
IN,916,34,3.71
R,2321,104,4.48
S,2960,127,4.29


In [21]:
# Final Data Profiling Checks

# 1. Numeric summary
display(
    wr_df.select_dtypes(include="number").describe().T
)

# 2. Important categorical columns
categorical_cols = [
    "pass_result",
    "route_ran",
    "receiver_alignment",
    "offense_formation",
    "team_coverage_type"
]

for col in categorical_cols:
    print(f"\n{col}")
    display(wr_df[col].value_counts(dropna=False))

,count,mean,std,min,25%,50%,75%,max
game_id,42806.0,2.024414e+09,830246.533026,2.023080e+09,2.024082e+09,2.024122e+09,2.025102e+09,2.026021e+09
play_id,42806.0,2.198868e+03,1254.884250,5.400000e+01,1.144000e+03,2.147000e+03,3.275750e+03,5.209000e+03
nfl_id,42806.0,5.668449e+04,879.659300,5.588400e+04,5.593900e+04,5.607500e+04,5.722300e+04,5.927800e+04
quarter,42806.0,2.575620e+00,1.118916,1.000000e+00,2.000000e+00,2.000000e+00,4.000000e+00,5.000000e+00
down,42806.0,2.021679e+00,0.867106,1.000000e+00,1.000000e+00,2.000000e+00,3.000000e+00,4.000000e+00
yards_to_go,42806.0,8.910106e+00,4.019047,1.000000e+00,6.000000e+00,1.000000e+01,1.000000e+01,4.000000e+01
yardline_number,42806.0,3.073139e+01,12.274811,1.000000e+00,2.200000e+01,3.200000e+01,4.100000e+01,5.000000e+01
pre_snap_home_score,42806.0,1.131143e+01,9.754783,0.000000e+00,3.000000e+00,1.000000e+01,1.700000e+01,7.000000e+01
pre_snap_visitor_score,42806.0,1.057894e+01,9.368372,0.000000e+00,3.000000e+00,9.000000e+00,1.700000e+01,5.100000e+01
pre_snap_home_team_win_probability,42806.0,5.332312e-01,0.307847,7.967445e-04,2.740411e-01,5.467556e-01,8.009583e-01,9.999500e-01



pass_result


pass_result
C     23603
I     13006
S      2960
R      2321
IN      916
Name: count, dtype: int64


route_ran


route_ran
GO        8533
HITCH     6980
CROSS     5374
IN        4829
POST      4310
OUT       4141
CORNER    3427
SLANT     2850
SCREEN    1169
FLAT      1110
WHEEL       59
ANGLE       24
Name: count, dtype: int64

...

receiver_alignment
3x1    18065
2x2    18016
3x2     4334
2x1     1952
4x1      358
1x1       44
NaN       21
4x0        6
3x0        4
3x3        3
4x2        1
5x2        1
4x3        1
Name: count, dtype: int64


offense_formation


offense_formation
SHOTGUN       31273
SINGLEBACK     5049
EMPTY          4660
PISTOL          999
I_FORM          758
NaN              28
JUMBO            27
WILDCAT          12
Name: count, dtype: int64


team_coverage_type


team_coverage_type
COVER_3_ZONE    13250
COVER_1_MAN      9259
COVER_4_ZONE     6716
COVER_2_ZONE     5650
COVER_6_ZONE     3762
COVER_0_MAN      1913
NaN              1525
COVER_2_MAN       578
PREVENT           153
Name: count, dtype: int64

## Final Data Profiling Summary — player_play.csv

### Dataset Overview
- Total records: 314,197
- Total columns: 64
- Unique players: 497
- Unique games: 1,001
- Unique game-play combinations: 114,954
- No duplicate rows or duplicate composite keys (`game_id`, `play_id`, `nfl_id`).

### Referential Integrity
- All 497 players in `player_play.csv` exist in `players.csv`.
- 13 players from `players.csv` have no player-play records.
- All 13 are present in `combine_results.csv`.
- One missing player belongs to the WR position.

### WR-Specific Findings
- Unique WR players: 105
- WR player-play records: 42,806
- Nine columns have 100% missing values for WRs.
- Eight of these columns are primarily related to defensive responsibilities.
- `rush_yards` is entirely missing for WRs and should be treated as unavailable in this dataset.

### Missing Values Findings
- Receiving metrics (`rec_yards`, `yards_after_catch`) are populated for completed receptions and are generally NULL otherwise.
- Two non-targeted WR records contain receiving-yard values and should be flagged for later validation.
- All 5,281 missing `separation_at_pass_forward` values occur in run or sack plays, where no forward pass is released.
- `cushion` has 2,075 missing values (4.85%), distributed across all pass-result categories without a clear outcome-specific pattern.
- Missingness in other features may be position-dependent or event-dependent.

### Data Quality Decisions
- Preserve the original dataset without modification.
- Do not automatically replace missing values with zero.
- Keep unresolved anomalies documented for preprocessing.
- Select features according to the final WR analysis objectives.

### Overall Conclusion
The dataset has consistent player identifiers and a valid player-play composite key. Most investigated missingness patterns are consistent with player responsibilities or play events. Remaining missingness and potential anomalies are documented for later investigation.

**Status: Initial Data Profiling Completed.**